# OCO 외곽 배율 비교: 29~34번
28번까지 저장한 OCO 결과를 읽어 K-MRC를 고정하고, 144mm 바깥 point가 있는 shot만 CPE를 재계산합니다. 기본 10~100%의 10개 배율은 PSM 추정에만 적용하며, M3S는 배율 적용 전 측정값에서 고정 보정 및 새 PSM을 뺀 metric_residual로 계산합니다. DB 조회·기존 29번 context 코드는 실행하지 않습니다. 이 파일만 새 커널에서 29~34번 순서대로 실행하거나 기존 28번 뒤에 복사하세요.


## 29. 입력·배율·메트릭 설정 / 저장된 27·28 결과 읽기


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import copy
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

ES_BASE_DIR = Path.cwd()  # C:\PHOTO_ANALYSIS\BASIC에 노트북 배치
ES_INPUT_DIR = ES_BASE_DIR / "output_decorrect_notebook"
ES_OUTPUT_DIR = ES_BASE_DIR / "output_oco_edge_scale_sweep"
ES_SCALES = [i / 10 for i in range(1, 11)]  # 10~100%, 10% 간격
ES_RADIUS_UM = 144000.0
ES_METRIC_AXES = ["x", "y"]  # X만 보고 싶으면 ["x"]
ES_STD_DDOF = 1  # 표본 표준편차. M3S = abs(mean) + 3*std
ES_SAVE_AFFECTED_POINTS = True  # 영향받는 shot의 point별 결과만 배율별 Parquet 저장
ES_BASELINE_ATOL_UM = 1e-9
ES_FILTERS = {}  # 예: {"LOT_ID": ["LOT_A"], "Wafer": [11]}
ES_WAFER_KEYS = ["LOT_ID", "Wafer"]
ES_SHOT_KEYS = ES_WAFER_KEYS + ["fcp_x", "fcp_y"]

if not ES_SCALES or any(not np.isfinite(v) or v < 0 for v in ES_SCALES):
    raise ValueError("비교 배율은 유한한 0 이상의 수여야 합니다.")
ES_SCALES = sorted(set(float(v) for v in ES_SCALES))
if ES_STD_DDOF not in [0, 1] or not set(ES_METRIC_AXES).issubset({"x", "y"}):
    raise ValueError("ES_STD_DDOF는 0/1, 방향은 x/y로 지정하세요.")
for folder in [ES_OUTPUT_DIR, ES_OUTPUT_DIR / "tables", ES_OUTPUT_DIR / "plots", ES_OUTPUT_DIR / "points"]:
    folder.mkdir(parents=True, exist_ok=True)


def es_read_full(stem):
    errors = []
    for suffix in [".parquet", ".xlsx"]:
        path = ES_INPUT_DIR / "stages" / (stem + suffix)
        if path.is_file():
            try:
                frame = pd.read_parquet(path) if suffix == ".parquet" else pd.read_excel(path)
                print(f"입력: {path.name} — {len(frame):,} rows")
                return frame, path
            except Exception as exc:
                errors.append(f"{path.name}: {exc}")
    raise FileNotFoundError(f"{stem} 전체 Parquet/Excel이 필요합니다. _sample 파일은 읽지 않습니다. " + " | ".join(errors))


def es_prepare_points(frame):
    required = ES_SHOT_KEYS + ["coordinate_X", "coordinate_Y", "raw_minus_total_fit_x", "raw_minus_total_fit_y", "CPE_fit_x", "CPE_fit_y", "real_residual_x", "real_residual_y", "measurement_scale", "X_reg_original", "Y_reg_original"]
    missing = [col for col in required if col not in frame]
    if missing:
        raise ValueError(f"27번 OCO 결과에 필수 컬럼이 없습니다: {missing}")
    out = frame.copy().reset_index(drop=True)
    for col, values in ES_FILTERS.items():
        if col not in out:
            raise ValueError(f"필터 컬럼 없음: {col}")
        out = out[out[col].isin(values if isinstance(values, (list, tuple, set)) else [values])].copy()
    out = out.reset_index(drop=True)
    if out.empty:
        raise ValueError("선택한 OCO point가 없습니다. 27번 결과 또는 ES_FILTERS를 확인하세요.")
    number_cols = required[2:] + ["radius", "wf_x", "wf_y", "raw_x", "raw_y", "X_reg", "Y_reg"]
    for col in number_cols:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if "radius" not in out:
        if "wf_x" in out and "wf_y" in out:
            out["radius"] = np.hypot(out["wf_x"], out["wf_y"])
        else:
            out["radius"] = np.hypot(out["fcp_x"] + out["coordinate_X"], out["fcp_y"] + out["coordinate_Y"])
    out["_es_outer"] = out["radius"].gt(ES_RADIUS_UM) & np.isfinite(out["radius"])
    valid_cols = ["raw_minus_total_fit_x", "raw_minus_total_fit_y", "coordinate_X", "coordinate_Y"]
    out["_es_fit_valid"] = np.isfinite(out[valid_cols].to_numpy(float)).all(axis=1)
    needed = out["_es_outer"] & out["_es_fit_valid"]
    if needed.any() and not np.isfinite(out.loc[needed, ["measurement_scale", "X_reg_original", "Y_reg_original"]].to_numpy(float)).all():
        raise ValueError("유효한 외곽 point의 원래 측정값 또는 기존 배율이 없습니다. 05~28 저장 자료를 확인하세요.")
    out["_es_affected"] = out["_es_outer"].groupby([out[col] for col in ES_SHOT_KEYS], observed=True, dropna=False).transform("any")
    return out


es_base_points, es_point_path = es_read_full("27_OCO_FINAL_POINTS")
es_settings_path = ES_INPUT_DIR / "extension_settings.json"
if not es_settings_path.is_file():
    raise FileNotFoundError(f"28번 설정 파일이 필요합니다: {es_settings_path}")
es_saved_settings = json.loads(es_settings_path.read_text(encoding="utf-8"))
es_base_points = es_prepare_points(es_base_points)
es_affected_base = es_base_points[es_base_points["_es_affected"]].copy()
es_wafer_index = es_base_points[ES_WAFER_KEYS].drop_duplicates().sort_values(ES_WAFER_KEYS).reset_index(drop=True)
es_wafer_index.insert(0, "wafer_index", np.arange(len(es_wafer_index)))
es_shot_index = es_affected_base[ES_SHOT_KEYS].drop_duplicates().sort_values(ES_SHOT_KEYS).reset_index(drop=True)
es_shot_index.insert(0, "shot_index", np.arange(len(es_shot_index)))
if not es_shot_index.empty:
    for col in ["DieX", "DieY", "photo_transn_seq"]:
        if col in es_affected_base:
            values = es_affected_base.groupby(ES_SHOT_KEYS, observed=True, dropna=False)[col].first().reset_index()
            es_shot_index = es_shot_index.merge(values, on=ES_SHOT_KEYS, how="left", validate="1:1")
    counts = es_affected_base.assign(n_outer=es_affected_base["_es_outer"].astype(int), n_outer_fit=(es_affected_base["_es_outer"] & es_affected_base["_es_fit_valid"]).astype(int))
    counts = counts.groupby(ES_SHOT_KEYS, observed=True, dropna=False).agg(n_points=("radius", "size"), n_outer_points=("n_outer", "sum"), n_outer_fit_points=("n_outer_fit", "sum")).reset_index()
    es_shot_index = es_shot_index.merge(counts, on=ES_SHOT_KEYS, how="left", validate="1:1")
else:
    for col in ["DieX", "DieY", "n_points", "n_outer_points", "n_outer_fit_points"]:
        es_shot_index[col] = pd.Series(dtype="Int64")
print(f"OCO wafer {len(es_wafer_index):,}, 전체 shot {len(es_base_points[ES_SHOT_KEYS].drop_duplicates()):,}, 재계산 shot {len(es_shot_index):,}")
print("K-MRC 고정. 반경 >144mm point의 배율은 CPE 회귀에만 적용; M3S는 배율 적용 전 측정값에서 새 CPE를 뺀 값으로 두 영역 집계.")
display(es_shot_index.head(10))


## 30. CPE38 모델·virtual grid 정의


In [ ]:
# 기존 22·23번 CPE 모델과 virtual grid. es_/ES_ 이름으로 분리해 기존 상태를 보존합니다.
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

ES_SPAN_MODE = "OVO3_38P_F2F"
ES_RIDGE_REGRESSION = True
ES_G_OPT_SPEC = 1.0
ES_USE_VIRTUAL_CHIP_POINTS = True   # VBA virtual grid: chip별 4 corner + center point
ES_G_OPT_TOLERANCE = 0.001          # VBA: G_opt_tolerlance = 0.001
ES_SPAN_TOLERANCE = 0.01            # VBA: span_tolerlance = 0.01
ES_MINIMUM_RIDGE = 0.001            # VBA: minimum_Ridge = 0.001
ES_ADD_RIDGE = 10.0                 # VBA: add_ridge = 10
ES_MAX_GOPT_RETURN = 50             # VBA: max_prevent_fallback_retrun = 50
ES_MAX_SPAN_ITER = 5000             # VBA: max_iter = 5000
ES_CD_TOL = 1e-6                    # VBA: tol = 0.000001
ES_CPE_EVAL_GRID_N = 10
ES_MIN_POINTS_GUIDE = 10
ES_G_OPTIMAL_TYPE = 1               # VBA information 기준: GOT1
ES_VIRTUAL_WAFER_RADIUS = 144000.0  # VBA virtual point keep 조건: R <= 144000               # VBA option D59: 1,2,3,4. 4 = inner OT2 / partial OT1
ES_USE_SPAN_BOX_CONSTRAINT = True   # VBA Balloon projected CD/PGD stage

def es_cpe_38para(rx: Sequence[float], ry: Sequence[float]) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(rx)),
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
        (rx ** 3 * ry) / 1e19, (rx ** 2 * ry ** 2) / 1e19, (rx * ry ** 3) / 1e19, (ry ** 4) / 1e19,
        (rx ** 3 * ry ** 2) / 1e23, (rx ** 2 * ry ** 3) / 1e23, (rx * ry ** 4) / 1e23, (ry ** 5) / 1e23,
        (rx ** 3 * ry ** 3) / 1e27, (rx ** 2 * ry ** 4) / 1e27,
        (rx ** 3 * ry ** 4) / 1e31,
    ]).T
    X_dy = np.vstack([
        np.ones(len(ry)),
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12,
        (ry ** 4) / 1e19, (ry ** 3 * rx) / 1e19, (ry ** 2 * rx ** 2) / 1e19,
        (ry ** 5) / 1e23, (ry ** 4 * rx) / 1e23, (ry ** 3 * rx ** 2) / 1e23,
        (ry ** 5 * rx) / 1e27, (ry ** 4 * rx ** 2) / 1e27,
    ]).T
    return X_dx, X_dy
ES_CPE38_KEYS_DX = [
    "RK1", "RK3", "RK5",
    "RK7", "RK9", "RK11",
    "RK13", "RK15", "RK17", "RK19",
    "RK23", "RK25", "RK27", "RK29",
    "RK35", "RK37", "RK39", "RK41",
    "RK49", "RK51",
    "RK65",
]
ES_CPE38_KEYS_DY = [
    "RK2", "RK4", "RK6",
    "RK8", "RK10", "RK12",
    "RK14", "RK16", "RK18",
    "RK22", "RK24", "RK26",
    "RK32", "RK34", "RK36",
    "RK46", "RK48",
]
ES_RK_SPAN_LIMITS_OVO3_38P_F2F = {
    'RK1': {'min': -0.1, 'max': 0.1},
    'RK2': {'min': -0.1, 'max': 0.1},
    'RK3': {'min': -2.0, 'max': 2.0},
    'RK4': {'min': -10.0, 'max': 10.0},
    'RK5': {'min': -5.0, 'max': 5.0},
    'RK6': {'min': -5.0, 'max': 5.0},
    'RK7': {'min': -0.1, 'max': 0.1},
    'RK8': {'min': -0.15, 'max': 0.15},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -0.5, 'max': 0.5},
    'RK11': {'min': -0.5, 'max': 0.5},
    'RK12': {'min': -0.03, 'max': 0.03},
    'RK13': {'min': -0.003, 'max': 0.003},
    'RK14': {'min': -0.015, 'max': 0.015},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -0.05, 'max': 0.05},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -0.023, 'max': 0.023},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}
ES_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT = {
    'RK1': {'min': -100.0, 'max': 100.0},
    'RK2': {'min': -100.0, 'max': 100.0},
    'RK3': {'min': -1000.0, 'max': 1000.0},
    'RK4': {'min': -1000.0, 'max': 1000.0},
    'RK5': {'min': -1000.0, 'max': 1000.0},
    'RK6': {'min': -1000.0, 'max': 1000.0},
    'RK7': {'min': -0.25, 'max': 0.25},
    'RK8': {'min': -10.0, 'max': 10.0},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -10.0, 'max': 10.0},
    'RK11': {'min': -10.0, 'max': 10.0},
    'RK12': {'min': -0.25, 'max': 0.25},
    'RK13': {'min': -0.025, 'max': 0.025},
    'RK14': {'min': -1.0, 'max': 1.0},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -1.0, 'max': 1.0},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -1.0, 'max': 1.0},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}

def es_get_active_span_limits(span_mode: str = "OVO3_38P_F2F") -> Dict[str, Dict[str, float]]:
    if span_mode == "OVO3_38P_HARDLIMIT":
        return ES_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT
    return ES_RK_SPAN_LIMITS_OVO3_38P_F2F
ES_RIDGE_VALVE_HYPER = {
    "RK1": 0.0, "RK2": 0.0,
    "RK3": 0.01, "RK4": 0.01, "RK5": 0.01, "RK6": 0.01,
    "RK7": 1.0, "RK8": 0.1, "RK9": 1.0, "RK10": 0.1,
    "RK11": 0.1, "RK12": 1.0,
    "RK13": 1.0, "RK14": 0.1, "RK15": 1.0, "RK16": 0.1,
    "RK17": 1.0, "RK18": 1.0, "RK19": 0.1, "RK20": 1.0,
}

def es__to_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def es_choose_base_lambda_scale(n_points: int) -> float:
    """VBA의 point가 적어도 skip하지 않는 방식을 Python식으로 반영."""
    if n_points >= 10:
        return 1.0
    if n_points >= 7:
        return 1e2
    if n_points >= 5:
        return 1e4
    if n_points >= 3:
        return 1e6
    return 1e8

def es_get_ridge_valves(coeff_names: Sequence[str]) -> np.ndarray:
    return np.array([ES_RIDGE_VALVE_HYPER.get(name, 1.0) for name in coeff_names], dtype=float)

def es_make_lambda_vector(base_lambda: float, coeff_names: Sequence[str]) -> np.ndarray:
    valves = es_get_ridge_valves(coeff_names)
    return float(base_lambda) * valves

def es_build_virtual_chip_points(
    step_pitch_x: float,
    step_pitch_y: float,
    chip_x_num: int,
    chip_y_num: int,
    round_decimals: int = 6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA virtual grid 재현:
    chip별 4 corner + center point를 shot-local coordinate로 생성하고,
    chip 경계에서 겹치는 corner는 dedup한다.
    예: chip_x_num=2, chip_y_num=1
      - 각 chip 5 point => 10 raw points
      - 가운데 공유 corner 2개 중복 제거
      - 최종 8 virtual points
    coordinate_X/Y가 shot center 기준이라고 보고,
    전체 shot 영역은 [-STEP_PITCH_X/2, +STEP_PITCH_X/2],
                  [-STEP_PITCH_Y/2, +STEP_PITCH_Y/2] 로 둔다.
    """
    chip_x_num = int(chip_x_num) if pd.notna(chip_x_num) and int(chip_x_num) > 0 else 1
    chip_y_num = int(chip_y_num) if pd.notna(chip_y_num) and int(chip_y_num) > 0 else 1
    sx = float(step_pitch_x)
    sy = float(step_pitch_y)
    x_edges = np.linspace(-sx / 2.0, sx / 2.0, chip_x_num + 1)
    y_edges = np.linspace(-sy / 2.0, sy / 2.0, chip_y_num + 1)
    pts = []
    for ix in range(chip_x_num):
        for iy in range(chip_y_num):
            x0, x1 = x_edges[ix], x_edges[ix + 1]
            y0, y1 = y_edges[iy], y_edges[iy + 1]
            xc = (x0 + x1) / 2.0
            yc = (y0 + y1) / 2.0
            pts.extend([
                (x0, y0), (x0, y1), (x1, y0), (x1, y1),
                (xc, yc),
            ])
    pts_unique = sorted({(round(x, round_decimals), round(y, round_decimals)) for x, y in pts})
    rx = np.array([p[0] for p in pts_unique], dtype=float)
    ry = np.array([p[1] for p in pts_unique], dtype=float)
    return rx, ry

def es_filter_virtual_points_by_wafer_radius(
    rx_v: Sequence[float],
    ry_v: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = ES_VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    VBA edge-shot virtual point filtering.
    virtual point의 wafer 좌표 R이 wafer_radius 이하인 point만 사용한다.
    R = sqrt((fcp_x + rx_v)^2 + (fcp_y + ry_v)^2)
    keep if R <= 144000
    """
    rx_v = np.asarray(rx_v, dtype=float)
    ry_v = np.asarray(ry_v, dtype=float)
    wx = float(fcp_x) + rx_v
    wy = float(fcp_y) + ry_v
    r = np.sqrt(wx * wx + wy * wy)
    keep = r <= float(wafer_radius)
    return rx_v[keep], ry_v[keep], r[keep]

def es_get_virtual_points_for_group(
    grp: pd.DataFrame,
    fallback_rx: Sequence[float],
    fallback_ry: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = ES_VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, str, int, int]:
    """
    1) full virtual point 생성: chip별 4 corner + center
    2) edge shot 처리: wafer 좌표 R <= wafer_radius point만 keep
    반환:
      rx_v, ry_v, virtual_source, n_virtual_full, n_virtual_kept
    """
    required = ["STEP_PITCH_X", "STEP_PITCH_Y", "CHIP_X_NUM", "CHIP_Y_NUM"]
    if all(c in grp.columns for c in required):
        vals = grp[required].dropna()
        if len(vals) > 0:
            row = vals.iloc[0]
            rx_full, ry_full = es_build_virtual_chip_points(
                step_pitch_x=row["STEP_PITCH_X"],
                step_pitch_y=row["STEP_PITCH_Y"],
                chip_x_num=row["CHIP_X_NUM"],
                chip_y_num=row["CHIP_Y_NUM"],
            )
            n_full = len(rx_full)
            rx_keep, ry_keep, _ = es_filter_virtual_points_by_wafer_radius(
                rx_full,
                ry_full,
                fcp_x=fcp_x,
                fcp_y=fcp_y,
                wafer_radius=wafer_radius,
            )
            if len(rx_keep) > 0:
                src = "chip_corner_center_wafer_clipped"
                return rx_keep, ry_keep, src, n_full, len(rx_keep)
            return rx_full, ry_full, "chip_corner_center_no_clip_points_fallback", n_full, n_full
    rx_v, ry_v = es_build_eval_grid_from_data(fallback_rx, fallback_ry, grid_n=ES_CPE_EVAL_GRID_N)
    return rx_v, ry_v, "measured_range_grid_fallback", len(rx_v), len(rx_v)


## 31. CPE38 solver·shot 회귀 / 28번 설정 복원


In [ ]:
# 기존 24~26번 solver와 shot별 회귀. 계산 방식은 그대로입니다.
def es_stable_inverse_or_pinv(A: np.ndarray) -> np.ndarray:
    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A)

def es_ridge_predict(X: np.ndarray, y: np.ndarray, lambdas: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X.T @ X + np.diag(lambdas)
    b = X.T @ y
    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(A) @ b
    return beta, X @ beta

def es_build_eval_grid_from_data(rx: Sequence[float], ry: Sequence[float], grid_n: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    rx_min, rx_max = np.nanmin(rx), np.nanmax(rx)
    ry_min, ry_max = np.nanmin(ry), np.nanmax(ry)
    if not np.isfinite(rx_min) or not np.isfinite(rx_max):
        rx_min, rx_max = -1.0, 1.0
    if not np.isfinite(ry_min) or not np.isfinite(ry_max):
        ry_min, ry_max = -1.0, 1.0
    if rx_min == rx_max:
        rx_min -= 1.0
        rx_max += 1.0
    if ry_min == ry_max:
        ry_min -= 1.0
        ry_max += 1.0
    gx = np.linspace(rx_min, rx_max, grid_n)
    gy = np.linspace(ry_min, ry_max, grid_n)
    RX, RY = np.meshgrid(gx, gy)
    return RX.ravel(), RY.ravel()

def es_gopt_value(
    X_meas: np.ndarray,
    X_eval: np.ndarray,
    lambdas: np.ndarray,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
) -> float:
    """
    VBA G_optimal_type 반영.
    - Type1: H H.T diag의 sqrt(max)
    - Type2: H matrix element의 signed max의 sqrt. 음수이면 999.0 반환.
    - Type3: X_eval inv(...) X_eval.T diag의 sqrt(max)
    - Type4: inner=Type2, partial=Type1
    """
    X_meas = np.asarray(X_meas, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X_meas.T @ X_meas + np.diag(lambdas)
    invA = es_stable_inverse_or_pinv(A)
    effective_type = gopt_type
    if gopt_type == 4:
        effective_type = 2 if inner_or_partial == "inner" else 1
    if effective_type == 1:
        H = X_eval @ invA @ X_meas.T
        diag_val = np.sum(H * H, axis=1)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    if effective_type == 2:
        H = X_eval @ invA @ X_meas.T
        max_val = float(np.nanmax(H))
        if max_val < 0:
            return 999.0
        return float(np.sqrt(max_val))
    if effective_type == 3:
        M = X_eval @ invA @ X_eval.T
        diag_val = np.diag(M)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    raise ValueError(f"Unsupported gopt_type: {gopt_type}")

def es_calc_contribution_spans(X_eval: np.ndarray, beta: np.ndarray, coeff_names: Sequence[str]) -> Dict[str, float]:
    spans = {}
    for i, name in enumerate(coeff_names):
        contrib = X_eval[:, i] * beta[i]
        spans[name] = float(np.nanmax(contrib) - np.nanmin(contrib))
    return spans

def es_find_span_violations(
    spans: Dict[str, float],
    span_limits: Dict[str, Dict[str, float]],
    tolerance: float = 0.01,
) -> List[Tuple[str, float, float, float]]:
    violations = []
    for name, span in spans.items():
        if name not in span_limits:
            continue
        lo = span_limits[name]["min"] - tolerance
        hi = span_limits[name]["max"] + tolerance
        if span < lo or span > hi:
            violations.append((name, span, span_limits[name]["min"], span_limits[name]["max"]))
    return violations

def es_build_beta_bounds_from_span(
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA의 min_Span_X/Y, max_Span_X/Y에 대응하는 box constraint 근사.
    contribution span = range(X_j * beta_j)가 span_limit 안에 들어오도록
    beta_j bound를 계산한다.
    """
    n = len(coeff_names)
    lo = np.full(n, -np.inf, dtype=float)
    hi = np.full(n, np.inf, dtype=float)
    for j, name in enumerate(coeff_names):
        if name not in span_limits:
            continue
        lim = span_limits[name]
        span_abs = max(abs(float(lim["min"])), abs(float(lim["max"])))
        xj_range = float(np.nanmax(X_eval[:, j]) - np.nanmin(X_eval[:, j]))
        if xj_range <= 0 or not np.isfinite(xj_range):
            continue
        b = span_abs / xj_range
        lo[j] = -b
        hi[j] = b
    return lo, hi

def es_projected_coordinate_descent_ridge(
    X: np.ndarray,
    y: np.ndarray,
    lambdas: np.ndarray,
    beta_init: Optional[np.ndarray] = None,
    lower: Optional[np.ndarray] = None,
    upper: Optional[np.ndarray] = None,
    tol: float = 1e-6,
    max_iter: int = 5000,
) -> Tuple[np.ndarray, np.ndarray, int, float]:
    """
    VBA ProjectedCoordinateDescentElasticNet(l1=0) 근사.
    solve min ||y-Xb||^2 + sum(lambda_j*b_j^2), s.t. lower <= b <= upper
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    n_param = X.shape[1]
    if beta_init is None:
        beta, _ = es_ridge_predict(X, y, lambdas)
    else:
        beta = np.asarray(beta_init, dtype=float).copy()
    if lower is None:
        lower = np.full(n_param, -np.inf)
    if upper is None:
        upper = np.full(n_param, np.inf)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    beta = np.clip(beta, lower, upper)
    residual = y - X @ beta
    col_norm2 = np.sum(X * X, axis=0)
    last_delta = np.inf
    for it in range(1, max_iter + 1):
        max_delta = 0.0
        for j in range(n_param):
            denom = col_norm2[j] + lambdas[j]
            if denom <= 0 or not np.isfinite(denom):
                continue
            residual += X[:, j] * beta[j]
            rho = float(X[:, j] @ residual)
            new_beta = rho / denom
            new_beta = min(max(new_beta, lower[j]), upper[j])
            delta = abs(new_beta - beta[j])
            beta[j] = new_beta
            residual -= X[:, j] * beta[j]
            if delta > max_delta:
                max_delta = delta
        last_delta = max_delta
        if max_delta < tol:
            break
    return beta, X @ beta, it, float(last_delta)

@dataclass
class ES_RidgeFitResult:
    beta: np.ndarray
    pred: np.ndarray
    lambdas: np.ndarray
    gopt: float
    spans: Dict[str, float]
    violations: List[Tuple[str, float, float, float]]
    gopt_iter: int
    span_iter: int
    method: str

def es_solve_vba_style_ridge_one_axis(
    X_fit: np.ndarray,
    y: np.ndarray,
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
    ridge_regression: bool = True,
    gopt_spec: float = 1.0,
    gopt_tolerance: float = 0.001,
    minimum_ridge: float = 0.001,
    add_ridge: float = 10.0,
    max_gopt_return: int = 50,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
    use_span_box_constraint: bool = True,
    span_tolerance: float = 0.01,
    cd_tol: float = 1e-6,
    max_span_iter: int = 5000,
    base_lambda_scale: float = 1.0,
    debug: bool = False,
) -> ES_RidgeFitResult:
    """
    VBA flow에 맞춘 한 방향(X or Y) fitting.
    - 초기 ridge는 0에 가까운 값이 아니라 VBA minimum_Ridge 기준 사용
    - Gopt가 spec 이상이면 ridge를 빠르게 키운 뒤 이분탐색으로 spec 근처를 찾음
    - span over가 남으면 projected coordinate descent로 coefficient를 box constraint 안에 넣음
    """
    X_fit = np.asarray(X_fit, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    y = np.asarray(y, dtype=float)
    n_param = X_fit.shape[1]
    if n_param != len(coeff_names):
        raise ValueError(f"n_param({n_param}) != len(coeff_names)({len(coeff_names)})")
    initial_lambda = 0.0 if not ridge_regression else 0.0
    lambdas = es_make_lambda_vector(initial_lambda, coeff_names)
    beta, pred = es_ridge_predict(X_fit, y, lambdas)
    g = es_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
    gopt_iter = 0
    method = "ols"
    if ridge_regression and (not np.isfinite(g) or g >= gopt_spec or g == 0):
        low_lambda = 0.0
        high_lambda = max(minimum_ridge * base_lambda_scale, minimum_ridge)
        high_lambda = max(high_lambda, minimum_ridge)
        high_lambdas = es_make_lambda_vector(high_lambda, coeff_names)
        high_g = es_gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        while (not np.isfinite(high_g) or high_g >= gopt_spec) and gopt_iter < max_gopt_return:
            low_lambda = high_lambda
            high_lambda = high_lambda * add_ridge
            high_lambdas = es_make_lambda_vector(high_lambda, coeff_names)
            high_g = es_gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
            gopt_iter += 1
            if debug:
                print(f"fast ridge: lambda={high_lambda:.6g}, gopt={high_g:.6g}")
        target_low = max(0.0, gopt_spec - gopt_tolerance)
        target_high = gopt_spec
        best_lambda = high_lambda
        best_g = high_g
        if np.isfinite(high_g) and high_g < gopt_spec:
            lo = low_lambda
            hi = high_lambda
            for _ in range(max_gopt_return):
                mid = (lo + hi) / 2.0
                mid_lambdas = es_make_lambda_vector(mid, coeff_names)
                mid_g = es_gopt_value(X_fit, X_eval, mid_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
                gopt_iter += 1
                if debug:
                    print(f"slow ridge: lambda={mid:.6g}, gopt={mid_g:.6g}")
                if target_low <= mid_g < target_high:
                    best_lambda = mid
                    best_g = mid_g
                    break
                if mid_g >= gopt_spec or not np.isfinite(mid_g):
                    lo = mid
                else:
                    hi = mid
                    best_lambda = mid
                    best_g = mid_g
                if abs(hi - lo) < max(minimum_ridge * 1e-3, 1e-12):
                    break
        lambdas = es_make_lambda_vector(best_lambda, coeff_names)
        beta, pred = es_ridge_predict(X_fit, y, lambdas)
        g = es_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt"
    spans = es_calc_contribution_spans(X_eval, beta, coeff_names)
    violations = es_find_span_violations(spans, span_limits, tolerance=span_tolerance)
    span_iter = 0
    if ridge_regression and use_span_box_constraint and len(violations) > 0:
        lower, upper = es_build_beta_bounds_from_span(X_eval, coeff_names, span_limits)
        beta_cd, pred_cd, span_iter, _ = es_projected_coordinate_descent_ridge(
            X_fit,
            y,
            lambdas,
            beta_init=beta,
            lower=lower,
            upper=upper,
            tol=cd_tol,
            max_iter=max_span_iter,
        )
        beta = beta_cd
        pred = pred_cd
        spans = es_calc_contribution_spans(X_eval, beta, coeff_names)
        violations = es_find_span_violations(spans, span_limits, tolerance=span_tolerance)
        g = es_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt_box_cd"
    return ES_RidgeFitResult(
        beta=beta,
        pred=pred,
        lambdas=lambdas,
        gopt=float(g),
        spans=spans,
        violations=violations,
        gopt_iter=int(gopt_iter),
        span_iter=int(span_iter),
        method=method,
    )

def es_determine_inner_or_partial(n_valid: int, n_virtual: int) -> str:
    """VBA: virtual_points = Vgrid_XY면 inner, 아니면 partial."""
    return "inner" if n_valid >= n_virtual else "partial"

def es_run_cpe38_vba_style_shotwise(
    df: pd.DataFrame,
    label: str = "ADI",
    wafer_cols: Tuple[str, str] = ("LOT_ID", "Wafer"),
    shot_center_cols: Tuple[str, str] = ("fcp_x", "fcp_y"),
    residual_cols: Tuple[str, str] = ("raw_minus_total_fit_x", "raw_minus_total_fit_y"),
    shot_cols: Tuple[str, str] = ("coordinate_X", "coordinate_Y"),
    span_mode: str = ES_SPAN_MODE,
    min_points_guide: int = ES_MIN_POINTS_GUIDE,
    eval_grid_n: int = ES_CPE_EVAL_GRID_N,
    ridge_regression: bool = ES_RIDGE_REGRESSION,
    gopt_spec: float = ES_G_OPT_SPEC,
    gopt_tolerance: float = ES_G_OPT_TOLERANCE,
    gopt_type: int = ES_G_OPTIMAL_TYPE,
    use_span_box_constraint: bool = ES_USE_SPAN_BOX_CONSTRAINT,
    debug: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    기존 018.py의 run_cpe38_ridge_shotwise_after_wkrk 대체 함수.
    Required input columns:
      LOT_ID, Wafer, fcp_x, fcp_y,
      raw_minus_total_fit_x, raw_minus_total_fit_y,
      coordinate_X, coordinate_Y
    """
    df = df.copy()
    required = list(wafer_cols) + list(shot_center_cols) + list(residual_cols) + list(shot_cols)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 필수 컬럼 누락: {missing}")
    es__to_numeric(df, list(shot_center_cols) + list(residual_cols) + list(shot_cols))
    out_cols = [
        "CPE_fit_x", "CPE_fit_y",
        "real_residual_x", "real_residual_y",
        "CPE_gopt_x", "CPE_gopt_y",
        "CPE_lambda_max_x", "CPE_lambda_max_y",
        "CPE_method_x", "CPE_method_y",
        "CPE_inner_or_partial",
    ]
    for c in out_cols:
        df[c] = np.nan if not c.startswith("CPE_method") and c != "CPE_inner_or_partial" else None
    span_limits = es_get_active_span_limits(span_mode)
    coef_rows = []
    summary_rows = []
    group_keys = list(wafer_cols) + list(shot_center_cols)
    for gkey, grp in df.groupby(group_keys, dropna=False, observed=True):
        valid_mask = (
            grp[residual_cols[0]].notna() &
            grp[residual_cols[1]].notna() &
            grp[shot_cols[0]].notna() &
            grp[shot_cols[1]].notna()
        )
        work = grp.loc[valid_mask].copy()
        n_valid = len(work)
        lot_id = gkey[0]
        wafer = gkey[1]
        fcp_x = gkey[2]
        fcp_y = gkey[3]
        if n_valid == 0:
            summary_rows.append({
                "LOT_ID": lot_id,
                "Wafer": wafer,
                "fcp_x": fcp_x,
                "fcp_y": fcp_y,
                "n_total": len(grp),
                "n_valid": 0,
                "few_points_flag": True,
                "status": "no_valid_points",
            })
            continue
        rx = work[shot_cols[0]].to_numpy(dtype=float)
        ry = work[shot_cols[1]].to_numpy(dtype=float)
        residual_x = work[residual_cols[0]].to_numpy(dtype=float)
        residual_y = work[residual_cols[1]].to_numpy(dtype=float)
        X_dx, X_dy = es_cpe_38para(rx, ry)
        if ES_USE_VIRTUAL_CHIP_POINTS:
            grid_rx, grid_ry, virtual_source, n_virtual_full, n_virtual = es_get_virtual_points_for_group(
                work,
                rx,
                ry,
                fcp_x=fcp_x,
                fcp_y=fcp_y,
                wafer_radius=ES_VIRTUAL_WAFER_RADIUS,
            )
        else:
            grid_rx, grid_ry = es_build_eval_grid_from_data(rx, ry, grid_n=eval_grid_n)
            virtual_source = "measured_range_grid"
            n_virtual_full = len(grid_rx)
            n_virtual = len(grid_rx)
        X_eval_dx, X_eval_dy = es_cpe_38para(grid_rx, grid_ry)
        inner_or_partial = es_determine_inner_or_partial(n_valid=n_valid, n_virtual=n_virtual)
        few_points_flag = n_valid < min_points_guide
        base_lambda_scale = es_choose_base_lambda_scale(n_valid)
        fit_x = es_solve_vba_style_ridge_one_axis(
            X_fit=X_dx,
            y=residual_x,
            X_eval=X_eval_dx,
            coeff_names=ES_CPE38_KEYS_DX,
            span_limits=span_limits,
            ridge_regression=ridge_regression,
            gopt_spec=gopt_spec,
            gopt_tolerance=gopt_tolerance,
            minimum_ridge=ES_MINIMUM_RIDGE,
            add_ridge=ES_ADD_RIDGE,
            max_gopt_return=ES_MAX_GOPT_RETURN,
            gopt_type=gopt_type,
            inner_or_partial=inner_or_partial,
            use_span_box_constraint=use_span_box_constraint,
            span_tolerance=ES_SPAN_TOLERANCE,
            cd_tol=ES_CD_TOL,
            max_span_iter=ES_MAX_SPAN_ITER,
            base_lambda_scale=base_lambda_scale,
            debug=debug,
        )
        fit_y = es_solve_vba_style_ridge_one_axis(
            X_fit=X_dy,
            y=residual_y,
            X_eval=X_eval_dy,
            coeff_names=ES_CPE38_KEYS_DY,
            span_limits=span_limits,
            ridge_regression=ridge_regression,
            gopt_spec=gopt_spec,
            gopt_tolerance=gopt_tolerance,
            minimum_ridge=ES_MINIMUM_RIDGE,
            add_ridge=ES_ADD_RIDGE,
            max_gopt_return=ES_MAX_GOPT_RETURN,
            gopt_type=gopt_type,
            inner_or_partial=inner_or_partial,
            use_span_box_constraint=use_span_box_constraint,
            span_tolerance=ES_SPAN_TOLERANCE,
            cd_tol=ES_CD_TOL,
            max_span_iter=ES_MAX_SPAN_ITER,
            base_lambda_scale=base_lambda_scale,
            debug=debug,
        )
        cpe_fit_x = fit_x.pred
        cpe_fit_y = fit_y.pred
        real_residual_x = residual_x - cpe_fit_x
        real_residual_y = residual_y - cpe_fit_y
        valid_idx = work.index
        df.loc[valid_idx, "CPE_fit_x"] = cpe_fit_x
        df.loc[valid_idx, "CPE_fit_y"] = cpe_fit_y
        df.loc[valid_idx, "real_residual_x"] = real_residual_x
        df.loc[valid_idx, "real_residual_y"] = real_residual_y
        df.loc[valid_idx, "CPE_gopt_x"] = fit_x.gopt
        df.loc[valid_idx, "CPE_gopt_y"] = fit_y.gopt
        df.loc[valid_idx, "CPE_lambda_max_x"] = float(np.nanmax(fit_x.lambdas))
        df.loc[valid_idx, "CPE_lambda_max_y"] = float(np.nanmax(fit_y.lambdas))
        df.loc[valid_idx, "CPE_method_x"] = fit_x.method
        df.loc[valid_idx, "CPE_method_y"] = fit_y.method
        df.loc[valid_idx, "CPE_inner_or_partial"] = inner_or_partial
        coef_row = {
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "fcp_x": fcp_x,
            "fcp_y": fcp_y,
            "n_total": len(grp),
            "n_valid": n_valid,
            "n_virtual": n_virtual,
            "n_virtual_full": n_virtual_full,
            "virtual_source": virtual_source,
            "inner_or_partial": inner_or_partial,
            "few_points_flag": few_points_flag,
            "base_lambda_scale": base_lambda_scale,
            "gopt_x": fit_x.gopt,
            "gopt_y": fit_y.gopt,
            "lambda_max_x": float(np.nanmax(fit_x.lambdas)),
            "lambda_max_y": float(np.nanmax(fit_y.lambdas)),
            "gopt_iter_x": fit_x.gopt_iter,
            "gopt_iter_y": fit_y.gopt_iter,
            "span_iter_x": fit_x.span_iter,
            "span_iter_y": fit_y.span_iter,
            "span_violation_count_x": len(fit_x.violations),
            "span_violation_count_y": len(fit_y.violations),
            "method_x": fit_x.method,
            "method_y": fit_y.method,
        }
        for name, val in zip(ES_CPE38_KEYS_DX, fit_x.beta):
            coef_row[f"{name}_X"] = val
            coef_row[f"span_{name}_X"] = fit_x.spans.get(name, np.nan)
        for name, val in zip(ES_CPE38_KEYS_DY, fit_y.beta):
            coef_row[f"{name}_Y"] = val
            coef_row[f"span_{name}_Y"] = fit_y.spans.get(name, np.nan)
        for name, value in zip(ES_CPE38_KEYS_DX, fit_x.lambdas):
            coef_row[f"lambda_{name}_X"] = float(value)
        for name, value in zip(ES_CPE38_KEYS_DY, fit_y.lambdas):
            coef_row[f"lambda_{name}_Y"] = float(value)
        coef_rows.append(coef_row)
        summary_rows.append({
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "fcp_x": fcp_x,
            "fcp_y": fcp_y,
            "n_total": len(grp),
            "n_valid": n_valid,
            "n_virtual": n_virtual,
            "n_virtual_full": n_virtual_full,
            "virtual_source": virtual_source,
            "inner_or_partial": inner_or_partial,
            "metro_ratio": n_valid / max(n_virtual, 1),
            "few_points_flag": few_points_flag,
            "base_lambda_scale": base_lambda_scale,
            "gopt_x": fit_x.gopt,
            "gopt_y": fit_y.gopt,
            "lambda_max_x": float(np.nanmax(fit_x.lambdas)),
            "lambda_max_y": float(np.nanmax(fit_y.lambdas)),
            "gopt_iter_x": fit_x.gopt_iter,
            "gopt_iter_y": fit_y.gopt_iter,
            "span_iter_x": fit_x.span_iter,
            "span_iter_y": fit_y.span_iter,
            "span_violation_count_x": len(fit_x.violations),
            "span_violation_count_y": len(fit_y.violations),
            "rms_before_x": float(np.sqrt(np.nanmean(residual_x ** 2))),
            "rms_before_y": float(np.sqrt(np.nanmean(residual_y ** 2))),
            "rms_after_x": float(np.sqrt(np.nanmean(real_residual_x ** 2))),
            "rms_after_y": float(np.sqrt(np.nanmean(real_residual_y ** 2))),
            "method_x": fit_x.method,
            "method_y": fit_y.method,
            "status": "ok",
        })
    coef_df = pd.DataFrame(coef_rows)
    summary_df = pd.DataFrame(summary_rows)
    print(f"[{label}] VBA-style CPE38 ridge 완료")
    print(f"  shot groups: {len(summary_df)}")
    print(f"  coef rows: {len(coef_df)}")
    if "few_points_flag" in summary_df.columns:
        print(f"  few points shots: {int(summary_df['few_points_flag'].sum())}")
    if "span_violation_count_x" in summary_df.columns:
        print(f"  remaining span violations X/Y: "
              f"{int(summary_df['span_violation_count_x'].sum())}/"
              f"{int(summary_df['span_violation_count_y'].sum())}")
    return df, coef_df, summary_df

# 28번에서 저장한 설정을 prefixed CPE solver에 복원합니다.
ES_CPE_OPTIONS = dict(es_saved_settings["cpe_options"])
es_global_settings = {
    "use_virtual_chip_points": "ES_USE_VIRTUAL_CHIP_POINTS", "virtual_wafer_radius_um": "ES_VIRTUAL_WAFER_RADIUS",
    "span_tolerance": "ES_SPAN_TOLERANCE", "minimum_ridge": "ES_MINIMUM_RIDGE", "add_ridge": "ES_ADD_RIDGE",
    "max_gopt_return": "ES_MAX_GOPT_RETURN", "max_span_iter": "ES_MAX_SPAN_ITER", "cd_tol": "ES_CD_TOL",
}
for key, variable in es_global_settings.items():
    if key not in es_saved_settings:
        raise ValueError(f"28번 실행 설정에 {key}가 없습니다. 28번을 다시 저장하세요.")
    globals()[variable] = es_saved_settings[key]
ES_RIDGE_VALVE_HYPER = copy.deepcopy(es_saved_settings["ridge_valves"])
if ES_CPE_OPTIONS["span_mode"] == "OVO3_38P_HARDLIMIT":
    ES_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT = copy.deepcopy(es_saved_settings["active_span_limits"])
else:
    ES_RK_SPAN_LIMITS_OVO3_38P_F2F = copy.deepcopy(es_saved_settings["active_span_limits"])
print("CPE 설정 복원:", ES_CPE_OPTIONS)


## 32. 고정 K-MRC에서 영향 shot 재계산 / wafer·shot M3S


In [ ]:
def es_metric_table(frame, keys, scale):
    skeleton = frame[keys].drop_duplicates().copy()
    results = []
    for region, mask in [("R_LE_144MM", frame["radius"].le(ES_RADIUS_UM) & np.isfinite(frame["radius"])), ("ALL_POINTS", pd.Series(True, index=frame.index))]:
        part = frame.loc[mask]
        for axis in ES_METRIC_AXES:
            result = skeleton.copy()
            if not part.empty:
                work = part[keys].copy()
                values = pd.to_numeric(part[f"metric_residual_{axis}"], errors="coerce") * 1000
                work["_value"] = values.where(np.isfinite(values))
                groups = work.groupby(keys, observed=True, dropna=False)
                stats = groups["_value"].agg(n_region_points="size", n_valid="count", mean_nm="mean")
                stats["sigma_nm"] = groups["_value"].std(ddof=ES_STD_DDOF)
                result = result.merge(stats.reset_index(), on=keys, how="left", validate="1:1")
            else:
                result = result.assign(n_region_points=0, n_valid=0, mean_nm=np.nan, sigma_nm=np.nan)
            result[["n_region_points", "n_valid"]] = result[["n_region_points", "n_valid"]].fillna(0).astype(int)
            result["abs_mean_nm"] = result["mean_nm"].abs()
            result["three_sigma_nm"] = 3 * result["sigma_nm"]
            result["M3S_nm"] = result["abs_mean_nm"] + result["three_sigma_nm"]
            result["metric_status"] = np.where(result.n_valid.eq(0), "no_valid_points", np.where(result.n_valid.le(ES_STD_DDOF), "too_few_points", "ok"))
            result["scale"], result["scale_pct"], result["axis"], result["region"] = scale, int(round(scale * 100)), axis, region
            result["metric_basis"] = "UNSCALED_MEASUREMENT"
            results.append(result)
    return pd.concat(results, ignore_index=True)


def es_scaled_input(base, scale):
    out = base.copy()
    outer = out["_es_outer"]
    for axis in ["x", "y"]:
        original = pd.to_numeric(out[f"{axis.upper()}_reg_original"], errors="coerce")
        delta = pd.Series(0., index=out.index)
        delta.loc[outer] = (scale - out.loc[outer, "measurement_scale"]) * original.loc[outer]
        for col in [f"raw_minus_total_fit_{axis}", f"raw_{axis}", f"{axis.upper()}_reg"]:
            if col in out:
                out[col] = pd.to_numeric(out[col], errors="coerce") + delta
    out.loc[outer, "measurement_scale"] = scale
    out["edge_sweep_scale"] = scale
    return out


def es_evaluate_original_measurement(fitted):
    """PSM은 배율 적용 입력으로 추정하고, 평가는 배율 적용 전 입력에서 같은 PSM을 뺀다."""
    out = fitted.copy()
    outer = out["_es_outer"]
    for axis in ["x", "y"]:
        restore = pd.Series(0., index=out.index)
        original = pd.to_numeric(out.loc[outer, f"{axis.upper()}_reg_original"], errors="coerce")
        scale = pd.to_numeric(out.loc[outer, "measurement_scale"], errors="coerce")
        restore.loc[outer] = (1. - scale) * original
        # 기존 MRC/PSM decorrection 및 K-MRC는 유지하고 측정값 배율만 100%로 복원한다.
        out[f"metric_input_{axis}"] = pd.to_numeric(out[f"raw_minus_total_fit_{axis}"], errors="coerce") + restore
        out[f"metric_residual_{axis}"] = out[f"metric_input_{axis}"] - pd.to_numeric(out[f"CPE_fit_{axis}"], errors="coerce")
    return out


def es_assert_baseline(recomputed, baseline):
    rows = []
    for col in ["CPE_fit_x", "CPE_fit_y", "real_residual_x", "real_residual_y"]:
        old = pd.to_numeric(baseline[col], errors="coerce").to_numpy(float)
        new = pd.to_numeric(recomputed[col], errors="coerce").to_numpy(float)
        passed = np.allclose(old, new, rtol=1e-7, atol=ES_BASELINE_ATOL_UM, equal_nan=True)
        valid = np.isfinite(old) & np.isfinite(new)
        rows.append(dict(column=col, n_compared=int(valid.sum()), max_abs_error_nm=float(np.max(np.abs(old[valid]-new[valid]))*1000) if valid.any() else np.nan, passed=bool(passed)))
    check = pd.DataFrame(rows)
    if not check.passed.all():
        check.to_excel(ES_OUTPUT_DIR / "Baseline_Parity_Failed.xlsx", index=False)
        raise ValueError("저장된 27번 결과와 CPE 재계산이 일치하지 않습니다. 28번 설정/입력 파일이 같은 실행 자료인지 확인하세요. Baseline_Parity_Failed.xlsx 참고.")
    return check


es_wafer_results, es_shot_results, es_coefficients, es_diagnostics, es_parity_rows = [], [], [], [], []
es_outer_scales = es_affected_base.loc[es_affected_base["_es_outer"], "measurement_scale"].dropna().unique()
es_matched_scale = float(es_outer_scales[0]) if len(es_outer_scales) == 1 and any(np.isclose(es_outer_scales[0], s, rtol=0, atol=1e-12) for s in ES_SCALES) else None
if not es_affected_base.empty and es_matched_scale is None:
    baseline_fit, _, _ = es_run_cpe38_vba_style_shotwise(es_affected_base, label="OCO_SAVED_BASELINE", **ES_CPE_OPTIONS)
    es_parity_rows.append(es_assert_baseline(baseline_fit, es_affected_base))
for scale in ES_SCALES:
    full = es_base_points[ES_SHOT_KEYS + ["radius", "_es_affected", "real_residual_x", "real_residual_y"]].copy()
    # 영향 없는 shot은 외곽 point가 없으므로 기존 residual이 곧 배율 적용 전 평가값이다.
    full = full.rename(columns={f"real_residual_{axis}": f"metric_residual_{axis}" for axis in ["x", "y"]})
    if not es_affected_base.empty:
        changed = es_scaled_input(es_affected_base, scale)
        fitted, coeff, diagnostic = es_run_cpe38_vba_style_shotwise(changed, label=f"OCO_EDGE_{scale:.0%}", **ES_CPE_OPTIONS)
        if es_matched_scale is not None and np.isclose(scale, es_matched_scale, rtol=0, atol=1e-12):
            es_parity_rows.append(es_assert_baseline(fitted, es_affected_base))
        fitted = es_evaluate_original_measurement(fitted)
        # Replace complete affected shots, including their inner points; unchanged shots reuse 27's results.
        for col in ["metric_residual_x", "metric_residual_y"]:
            full.loc[fitted.index, col] = fitted[col].to_numpy()
        for table, store in [(coeff, es_coefficients), (diagnostic, es_diagnostics)]:
            table = table.reindex(columns=list(dict.fromkeys(ES_SHOT_KEYS + list(table.columns))))
            table["scale"], table["scale_pct"] = scale, int(round(scale*100))
            store.append(table)
        if ES_SAVE_AFFECTED_POINTS:
            selected = ES_SHOT_KEYS + [col for col in ["photo_transn_seq", "DieX", "DieY", "TEST", "GROUP", "coordinate_X", "coordinate_Y", "radius", "measurement_source", "measurement_scale", "X_reg_original", "Y_reg_original", "raw_x", "raw_y", "wk_fit_x", "wk_fit_y", "rk_fit_x", "rk_fit_y", "total_fit_x", "total_fit_y", "raw_minus_total_fit_x", "raw_minus_total_fit_y", "CPE_fit_x", "CPE_fit_y", "real_residual_x", "real_residual_y", "metric_input_x", "metric_input_y", "metric_residual_x", "metric_residual_y"] if col in fitted]
            fitted[selected].to_parquet(ES_OUTPUT_DIR / "points" / f"Affected_points_scale_{int(round(scale*100)):03d}.parquet", index=False)
    wafer = es_metric_table(full, ES_WAFER_KEYS, scale)
    wafer = wafer.merge(es_wafer_index, on=ES_WAFER_KEYS, how="left", validate="m:1")
    shot = es_metric_table(full.loc[full["_es_affected"]], ES_SHOT_KEYS, scale)
    if not shot.empty:
        shot = shot.merge(es_shot_index, on=ES_SHOT_KEYS, how="left", validate="m:1")
    es_wafer_results.append(wafer)
    es_shot_results.append(shot)
    print(f"{scale:.0%}: wafer metric {len(wafer):,} rows, 영향 shot metric {len(shot):,} rows")
es_wafer_metrics = pd.concat(es_wafer_results, ignore_index=True)
es_shot_metrics = pd.concat(es_shot_results, ignore_index=True)
es_baseline_parity = pd.concat(es_parity_rows, ignore_index=True) if es_parity_rows else pd.DataFrame([dict(column="no_affected_shots", n_compared=0, max_abs_error_nm=0., passed=True)])
es_cpe_coefficients = pd.concat(es_coefficients, ignore_index=True) if es_coefficients else pd.DataFrame(columns=ES_SHOT_KEYS + ["scale", "scale_pct"])
es_cpe_diagnostics = pd.concat(es_diagnostics, ignore_index=True) if es_diagnostics else pd.DataFrame(columns=ES_SHOT_KEYS + ["scale", "scale_pct"])
display(es_baseline_parity)
display(es_wafer_metrics.head(12))


## 33. Wafer·영향 shot 비교 그래프 / 민감도


In [ ]:
from matplotlib.ticker import MaxNLocator

def es_sensitivity(metrics, keys):
    groups = keys + ["axis", "region"]
    if metrics.empty:
        return pd.DataFrame(columns=groups + ["min_M3S_nm", "max_M3S_nm", "range_M3S_nm", "best_scale_pct", "M3S_first_nm", "M3S_last_nm", "delta_last_minus_first_nm"])
    valid = metrics[metrics["M3S_nm"].notna()]
    result = metrics[groups].drop_duplicates().copy()
    if valid.empty:
        return result.assign(min_M3S_nm=np.nan, max_M3S_nm=np.nan, range_M3S_nm=np.nan, best_scale_pct=np.nan)
    stats = valid.groupby(groups, observed=True, dropna=False).agg(min_M3S_nm=("M3S_nm", "min"), max_M3S_nm=("M3S_nm", "max")).reset_index()
    best_idx = valid.groupby(groups, observed=True, dropna=False)["M3S_nm"].idxmin()
    best = valid.loc[best_idx, groups + ["scale_pct"]].rename(columns={"scale_pct": "best_scale_pct"})
    result = result.merge(stats, on=groups, how="left", validate="1:1").merge(best, on=groups, how="left", validate="1:1")
    result["range_M3S_nm"] = result["max_M3S_nm"] - result["min_M3S_nm"]
    for scale, name in [(ES_SCALES[0], "M3S_first_nm"), (ES_SCALES[-1], "M3S_last_nm")]:
        selected = metrics[metrics.scale.eq(scale)][groups + ["M3S_nm"]].rename(columns={"M3S_nm": name})
        result = result.merge(selected, on=groups, how="left", validate="1:1")
    result["delta_last_minus_first_nm"] = result["M3S_last_nm"] - result["M3S_first_nm"]
    return result


es_wafer_sensitivity = es_sensitivity(es_wafer_metrics, ES_WAFER_KEYS)
es_shot_sensitivity = es_sensitivity(es_shot_metrics, ES_SHOT_KEYS)
if not es_shot_sensitivity.empty:
    es_shot_sensitivity = es_shot_sensitivity.merge(es_shot_index, on=ES_SHOT_KEYS, how="left", validate="m:1")
es_plot_paths = []
es_colors = dict(zip(ES_SCALES, plt.get_cmap("tab10").colors))
es_region_titles = {"R_LE_144MM": "R <= 144 mm (current metric)", "ALL_POINTS": "All measured points"}


def es_save_plot(fig, stem):
    for extension in ["png", "pdf"]:
        path = ES_OUTPUT_DIR / "plots" / f"{stem}.{extension}"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        es_plot_paths.append(path)
    display(fig)
    plt.close(fig)


for axis in ES_METRIC_AXES:
    fig, panels = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    for panel, region in zip(panels, es_region_titles):
        for scale in ES_SCALES:
            part = es_wafer_metrics[es_wafer_metrics.axis.eq(axis) & es_wafer_metrics.region.eq(region) & es_wafer_metrics.scale.eq(scale)].sort_values("wafer_index")
            panel.plot(part.wafer_index, part.M3S_nm, ".-", markersize=3, linewidth=1, color=es_colors[scale], label=f"{scale:.0%}")
        panel.set_title(f"OCO {axis.upper()}: wafer M3S | {es_region_titles[region]} | unscaled evaluation")
        panel.set_ylabel("|mean| + 3 sigma [nm]")
        panel.grid(alpha=.25)
        panel.legend(title="Outer-point scale for CPE fit", ncol=len(ES_SCALES))
    panels[-1].set_xlabel("Wafer index (see Wafer_index sheet)")
    panels[-1].xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.tight_layout()
    es_save_plot(fig, f"01_Wafer_M3S_{axis.upper()}")

    fig, panels = plt.subplots(2, 1, figsize=(18, 8), sharex=True)
    for panel, region in zip(panels, es_region_titles):
        if es_shot_metrics.empty:
            panel.text(.5, .5, "No shots contain measured points beyond 144 mm", ha="center", transform=panel.transAxes)
        else:
            for scale in ES_SCALES:
                part = es_shot_metrics[es_shot_metrics.axis.eq(axis) & es_shot_metrics.region.eq(region) & es_shot_metrics.scale.eq(scale)].sort_values("shot_index")
                panel.plot(part.shot_index, part.M3S_nm, ".-", markersize=2, linewidth=.8, color=es_colors[scale], label=f"{scale:.0%}")
            panel.legend(title="Outer-point scale for CPE fit", ncol=len(ES_SCALES))
        panel.set_title(f"OCO {axis.upper()}: affected shots | {es_region_titles[region]} | unscaled evaluation")
        panel.set_ylabel("|mean| + 3 sigma [nm]")
        panel.grid(alpha=.25)
    panels[-1].set_xlabel("Affected shot index (see Affected_shot_index: LOT / wafer / DieX / DieY)")
    panels[-1].xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.tight_layout()
    es_save_plot(fig, f"02_Affected_Shot_M3S_{axis.upper()}")

    part = es_shot_sensitivity[es_shot_sensitivity.axis.eq(axis) & es_shot_sensitivity.region.eq("R_LE_144MM")] if not es_shot_sensitivity.empty else es_shot_sensitivity
    part = part.dropna(subset=["range_M3S_nm"]).sort_values("range_M3S_nm", ascending=False).head(25)
    if not part.empty:
        labels = [f"{row.LOT_ID} / W{row.Wafer} / Die({row.DieX},{row.DieY})" for row in part.itertuples()]
        fig, panel = plt.subplots(figsize=(13, max(4, len(part)*.28)))
        panel.barh(np.arange(len(part)), part.range_M3S_nm, color="steelblue")
        panel.set_yticks(np.arange(len(part)), labels)
        panel.invert_yaxis()
        panel.set_xlabel(f"M3S max - min across {len(ES_SCALES)} scales [nm]")
        panel.set_title(f"OCO {axis.upper()}: most sensitive shots | R <= 144 mm")
        panel.grid(axis="x", alpha=.25)
        fig.tight_layout()
        es_save_plot(fig, f"03_Shot_Sensitivity_Inner_{axis.upper()}")


## 34. Excel·전체 Parquet·실행 설정 저장


In [ ]:
es_notes = pd.DataFrame([
    ["입력", str(es_point_path)],
    ["계산", "반경 >144mm 측정점의 원래 X/Y에 지정한 배율 적용. 저장된 raw_minus_total_fit에 배율 차이만 더함. K-MRC 및 기존 MRC/PSM fitting은 고정."],
    ["CPE 재계산", "외곽 측정점이 포함된 shot의 전체 point만 CPE38 재회귀. 그 외 shot의 최종 결과는 저장값 재사용."],
    ["Wafer_M3S", "모든 분석 wafer. 영향 shot과 영향 없는 shot을 모두 합친 metric_residual의 통계. 평가 측정값에는 배율 미적용."],
    ["Affected_shot_M3S", "144mm 바깥 측정점이 하나라도 있는 shot만 수록. 안쪽 point도 해당 shot의 새 CPE 결과 사용."],
    ["M3S", f"abs(mean(metric_residual)) + 3*std(metric_residual), std ddof={ES_STD_DDOF}, nm 단위. mean(abs(residual))와 다름."],
    ["평가 입력", "metric_input = raw_minus_total_fit + (1-measurement_scale)*원래 측정값 (144mm 초과 point만 복원). 기존 decorrection 및 K-MRC는 유지."],
    ["평가 residual", "metric_residual = metric_input - CPE_fit. 안쪽/전체 및 wafer/shot M3S 모두 이 컬럼 사용. 배율은 PSM 추정에만 영향."],
    ["회귀 residual", "real_residual = raw_minus_total_fit - CPE_fit. 배율이 적용된 회귀 입력 기준 진단값으로 보존하며 M3S에는 사용하지 않음."],
    ["영역", "R_LE_144MM: 실제 측정점 반경 <=144mm (현재 사용 메트릭). ALL_POINTS: 모든 측정점. 회귀 입력은 영역별로 자르지 않음."],
    ["유효 point", "통계는 유한한 final residual만 사용. 안쪽 point 0개 또는 표준편차 계산 불가 시 M3S는 NA이며 0으로 채우지 않음."],
    ["배율", "각 외곽 point의 measurement_scale과 X_reg_original/Y_reg_original에서 복원. raw를 단순히 배율로 나누거나 곱하지 않음."],
    ["고정 요소", "좌표·회귀 point·K-MRC·기존 MRC/PSM·CPE 모델 및 설정. 배율만 변경."],
    ["best_scale_pct", f"지정한 {len(ES_SCALES)}개 후보 중 M3S가 최소인 배율. 동률이면 낮은 배율을 표시."],
    ["현재 반경 미상 point", str(int(es_base_points.radius.isna().sum()))],
], columns=["item", "description"])
es_tables = {
    "Wafer_M3S": es_wafer_metrics, "Affected_shot_M3S": es_shot_metrics,
    "Wafer_sensitivity": es_wafer_sensitivity, "Shot_sensitivity": es_shot_sensitivity,
    "Wafer_index": es_wafer_index, "Affected_shot_index": es_shot_index,
    "Baseline_parity": es_baseline_parity, "Notes": es_notes,
}


def es_excel_safe(frame):
    out = frame.copy()
    for col in out.columns:
        if col in ["photo_transn_seq", "source_lot_transn_seq", "apc_hist_index_no", "apc_trocs_hist_index_no"]:
            out[col] = out[col].map(lambda v: str(int(v)) if pd.notna(v) else "")
    return out


es_excel_path = ES_OUTPUT_DIR / "OCO_Edge_Scale_Sweep.xlsx"
with pd.ExcelWriter(es_excel_path, engine="openpyxl") as writer:
    for name, table in es_tables.items():
        table.to_parquet(ES_OUTPUT_DIR / "tables" / f"{name}.parquet", index=False)
        export = es_excel_safe(table)
        for offset in range(0, max(1, len(export)), 1048575):
            suffix = "" if len(export) <= 1048575 else f"_{offset//1048575+1}"
            export.iloc[offset:offset+1048575].to_excel(writer, sheet_name=name[:31-len(suffix)] + suffix, index=False)
for name, table in [("Affected_CPE_Coefficients", es_cpe_coefficients), ("Affected_CPE_Diagnostics", es_cpe_diagnostics)]:
    table.to_parquet(ES_OUTPUT_DIR / "tables" / f"{name}.parquet", index=False)
es_run_settings = dict(input_points=str(es_point_path), input_settings=str(es_settings_path), scales=ES_SCALES, metric_axes=ES_METRIC_AXES, metric_radius_um=ES_RADIUS_UM,
                       std_ddof=ES_STD_DDOF, metric_basis="UNSCALED_MEASUREMENT", metric_residual_columns=["metric_residual_x", "metric_residual_y"],
                       fixed_kmrc=True, recompute_only_affected_shots=True, n_input_points=len(es_base_points), n_affected_shots=len(es_shot_index), cpe_settings=es_saved_settings)
(ES_OUTPUT_DIR / "sweep_settings.json").write_text(json.dumps(es_run_settings, ensure_ascii=False, indent=2), encoding="utf-8")
display(FileLink(os.path.relpath(es_excel_path, Path.cwd()), result_html_prefix="Wafer/shot M3S Excel: "))
for path in es_plot_paths:
    if path.suffix == ".png":
        display(FileLink(os.path.relpath(path, Path.cwd()), result_html_prefix="그래프: "))
print("배율 비교 완료:", ES_OUTPUT_DIR)
